In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_5_day_2.csv',
    'prices_round_5_day_3.csv',
    'prices_round_5_day_4.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [18]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress

# --- 1. CONFIGURATION ---
JUMP_THRESHOLD_SD = 1.5  # Sensitivity for the 'Step Jump'
CONFIRM_TICKS = 1000  # Ticks outside bounds to trigger snap
PULLBACK_STRENGTH = 2e-8 # Constant gravity toward black line
SNAP_RATIO = 1        # Snap to 75% of the move to avoid overshooting spikes
VOL_WINDOW = 800

# --- 2. THE DAMPED PLATEAU FILTER ---
def apply_damped_plateau_filter(series, slope, intercept, jump_sd, confirm_period, pullback, snap_ratio):
    local_means = []
    current_local_mean = series.iloc[0]
    outside_counter = 0
    
    # Pre-calculate rolling volatility
    rolling_vol = series.diff().rolling(VOL_WINDOW).std().ffill().bfill().clip(lower=1.0)
    
    for i, val in enumerate(series):
        # A) Global Anchor Projection
        global_target = intercept + (slope * i)
        
        # B) Continuous Reversion (The "Slow Pull")
        current_local_mean += slope # Respect the global trend drift
        current_local_mean -= pullback * (current_local_mean - global_target)
        
        # C) Step Jump Logic with Damping
        sigma = rolling_vol.iloc[i]
        upper_limit = current_local_mean + (jump_sd * sigma)
        lower_limit = current_local_mean - (jump_sd * sigma)
        
        if val > upper_limit or val < lower_limit:
            outside_counter += 1
        else:
            outside_counter = 0
            
        if outside_counter >= confirm_period:
            # DAMPED SNAP: Move most of the way, but not all the way.
            # current_local_mean = (Old Mean * 25%) + (New Price * 75%)
            current_local_mean = (current_local_mean * (1 - snap_ratio)) + (val * snap_ratio)
            outside_counter = 0
            
        local_means.append(current_local_mean)
        
    return pd.Series(local_means, index=series.index)

# --- 3. DATA & EXECUTION ---
df_full = df_total[df_total['product'] == "SNACKPACK_CHOCOLATE"].copy().sort_values(['day', 'timestamp'])
df_full['global_tick'] = range(len(df_full))
df_full['mid_price'] = df_full['mid_price'].replace(0, np.nan).ffill()

# Global slope calculation
slope, intercept, _, _, _ = linregress(df_full['global_tick'], df_full['mid_price'])
slope = 0

df_full['local_mean'] = apply_damped_plateau_filter(
    df_full['mid_price'], slope, intercept, 
    JUMP_THRESHOLD_SD, CONFIRM_TICKS, PULLBACK_STRENGTH, SNAP_RATIO
)

# Z-Score Calculation
df_full['err'] = df_full['mid_price'] - df_full['local_mean']
df_full['rolling_std'] = df_full['err'].rolling(500).std().clip(lower=1.0)
df_full['z_score'] = df_full['err'] / df_full['rolling_std']

# --- 4. VISUALIZATION ---
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# Subplot 1: Price and Means
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=df_full['mid_price'], 
                         line=dict(color='lightgrey', width=1), name='Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=df_full['local_mean'], 
                         line=dict(color='red', width=2), name='Damped Plateau Mean'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=intercept + slope*df_full['global_tick'], 
                         line=dict(color='black', dash='dot'), name='Global Mean'), row=1, col=1)

# Subplot 2: Signal
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=df_full['z_score'], 
                         line=dict(color='blue'), name='Z-Score'), row=2, col=1)
fig.add_hline(y=2.0, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-2.0, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(height=800, title="Damped Plateau: Partial Snaps + Constant Reversion", template="plotly_white")
fig.show()

In [19]:
import numpy as np
import pandas as pd
from scipy.stats import linregress
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Jump-aware linear mean filter (posterior probability of new-mean regime)
def apply_jump_linear_mean_filter(
    series: pd.Series,
    slope: float,
    intercept: float,
    jump_prior: float = 0.005,
    jump_threshold: float = 0.82,
    jump_var_mult: float = 16.0,
    level_alpha: float = 0.05,
    slope_alpha: float = 0.04,
    var_alpha: float = 0.12,
    jump_confirm_ticks: int = 3,
    jump_cooldown_ticks: int = 60,
    jump_snap: float = 0.75,
):
    values = series.to_numpy(dtype=float)
    n = len(values)

    level = values[0]
    drift = float(slope)
    var = max(np.nanstd(np.diff(values[: min(300, n)])) ** 2, 1.0)

    local_mean = np.zeros(n)
    p_jump_arr = np.zeros(n)
    resid_arr = np.zeros(n)

    local_mean[0] = level
    jump_counter = 0
    cooldown = 0

    for i in range(1, n):
        x = values[i]

        # Global linear anchor (optional weak pull)
        global_target = intercept + slope * i

        # Prediction step
        pred = level + drift
        resid = x - pred

        base_var = max(var, 1e-8)
        jump_var = jump_var_mult * base_var

        # Likelihoods under: no jump vs jump
        like_no_jump = np.exp(-0.5 * resid * resid / base_var) / np.sqrt(2.0 * np.pi * base_var)
        like_jump = np.exp(-0.5 * resid * resid / jump_var) / np.sqrt(2.0 * np.pi * jump_var)

        num = jump_prior * like_jump
        den = num + (1.0 - jump_prior) * like_no_jump
        p_jump = num / den if den > 1e-16 else jump_prior

        prev_level = level

        # Jump needs confirmation and is rate-limited by cooldown.
        if p_jump >= jump_threshold and cooldown == 0:
            jump_counter += 1
        else:
            jump_counter = 0

        confirmed_jump = jump_counter >= max(1, int(jump_confirm_ticks))

        if confirmed_jump:
            # Regime jump detected: partial snap and damp drift.
            level = pred + float(jump_snap) * resid
            drift = 0.5 * drift
            cooldown = max(0, int(jump_cooldown_ticks))
            jump_counter = 0
        else:
            # Normal regime: conservative update so mean does not clone price.
            k = min(0.12, level_alpha * (1.0 - p_jump))
            level = pred + k * resid
            drift = (1.0 - slope_alpha) * drift + slope_alpha * (level - prev_level)

        # Optional weak attraction to global linear trend (kept tiny)
        level = 0.999 * level + 0.001 * global_target

        post_resid = x - level
        var = (1.0 - var_alpha) * var + var_alpha * (post_resid * post_resid)
        var = max(var, 1e-8)

        if cooldown > 0:
            cooldown -= 1

        local_mean[i] = level
        p_jump_arr[i] = p_jump
        resid_arr[i] = post_resid

    out = pd.DataFrame(index=series.index)
    out["local_mean_jump"] = local_mean
    out["p_jump"] = p_jump_arr
    out["err_jump"] = resid_arr
    return out


# --- A/B on same product as your plateau filter ---
df_j = df_total[df_total["product"] == "SNACKPACK_CHOCOLATE"].copy().sort_values(["day", "timestamp"])
df_j["global_tick"] = np.arange(len(df_j))
df_j["mid_price"] = df_j["mid_price"].replace(0, np.nan).ffill().bfill()

# Keep slope choice explicit for experiments
slope_j, intercept_j, _, _, _ = linregress(df_j["global_tick"], df_j["mid_price"])
slope_j = 0.0

jump_out = apply_jump_linear_mean_filter(
    df_j["mid_price"],
    slope=slope_j,
    intercept=intercept_j,
    jump_prior=0.000001,
    jump_threshold=0.82,
    jump_var_mult=16.0,
    level_alpha=0.05,
    slope_alpha=0.04,
    var_alpha=0.12,
    jump_confirm_ticks=400,
    jump_cooldown_ticks=60,
    jump_snap=0.75,
)

df_j = df_j.join(jump_out)
df_j["rolling_std_jump"] = df_j["err_jump"].rolling(500).std().clip(lower=1.0)
df_j["z_jump"] = df_j["err_jump"] / df_j["rolling_std_jump"]

# If the previous cell created plateau columns, compare directly.
has_plateau = "local_mean" in df_j.columns and "z_score" in df_j.columns

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.5, 0.25, 0.25],
)

# Row 1: price + means
fig.add_trace(
    go.Scatter(
        x=df_j["global_tick"],
        y=df_j["mid_price"],
        line=dict(color="lightgray", width=1),
        opacity=0.55,
        name="Price",
    ),
    row=1,
    col=1,
)
if has_plateau:
    fig.add_trace(
        go.Scatter(x=df_j["global_tick"], y=df_j["local_mean"], line=dict(color="red", width=2), name="Plateau mean"),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(
        x=df_j["global_tick"],
        y=df_j["local_mean_jump"],
        line=dict(color="green", width=3),
        name="Jump-linear mean",
    ),
    row=1,
    col=1,
)

# Row 2: z-scores
if has_plateau:
    fig.add_trace(
        go.Scatter(x=df_j["global_tick"], y=df_j["z_score"], line=dict(color="red", width=1), name="z plateau"),
        row=2,
        col=1,
    )
fig.add_trace(
    go.Scatter(x=df_j["global_tick"], y=df_j["z_jump"], line=dict(color="blue", width=1), name="z jump-linear"),
    row=2,
    col=1,
)
fig.add_hline(y=2.0, line_dash="dash", line_color="black", row=2, col=1)
fig.add_hline(y=-2.0, line_dash="dash", line_color="black", row=2, col=1)

# Row 3: posterior jump probability
fig.add_trace(
    go.Scatter(x=df_j["global_tick"], y=df_j["p_jump"], line=dict(color="purple", width=1), name="p_jump"),
    row=3,
    col=1,
)
fig.add_hline(y=0.55, line_dash="dot", line_color="purple", row=3, col=1)

fig.update_layout(
    height=950,
    template="plotly_white",
    title="A/B: Damped Plateau vs Jump-Probability Linear Mean",
)
fig.show()

print("Quick diagnostics")
print("z_jump mean abs:", float(np.nanmean(np.abs(df_j["z_jump"]))))
print("jump events (p_jump > 0.82):", int((df_j["p_jump"] > 0.82).sum()))
print("jump event rate:", float((df_j["p_jump"] > 0.82).mean()))
print("mean abs(price - mean):", float(np.nanmean(np.abs(df_j["mid_price"] - df_j["local_mean_jump"]))))
print("corr(price, mean):", float(df_j[["mid_price", "local_mean_jump"]].corr().iloc[0, 1]))

Quick diagnostics
z_jump mean abs: 0.8164114244643801
jump events (p_jump > 0.82): 0
jump event rate: 0.0
mean abs(price - mean): 15.312565824841883
corr(price, mean): 0.9954899692246854


In [20]:
# Quick parameter sweep for iteration
# Uses a simple objective: stronger mean-reversion structure in residuals
# => more negative lag-1 autocorrelation + stable z distribution.

rows = []
for jp in [0.01, 0.03, 0.06]:
    for jt in [0.45, 0.55, 0.70]:
        for jvm in [4.0, 9.0, 16.0]:
            out = apply_jump_linear_mean_filter(
                df_j["mid_price"],
                slope=slope_j,
                intercept=intercept_j,
                jump_prior=jp,
                jump_threshold=jt,
                jump_var_mult=jvm,
                level_alpha=0.20,
                slope_alpha=0.08,
                var_alpha=0.07,
            )
            err = out["err_jump"]
            z = err / err.rolling(500).std().clip(lower=1.0)

            lag1_autocorr = float(err.autocorr(lag=1))
            jump_rate = float((out["p_jump"] > jt).mean())
            z95 = float(np.nanquantile(np.abs(z), 0.95))

            # lower is better for this score
            score = lag1_autocorr + 0.05 * abs(jump_rate - 0.03) + 0.01 * abs(z95 - 2.8)

            rows.append({
                "jump_prior": jp,
                "jump_threshold": jt,
                "jump_var_mult": jvm,
                "lag1_autocorr": lag1_autocorr,
                "jump_rate": jump_rate,
                "z95_abs": z95,
                "score": score,
            })

sweep = pd.DataFrame(rows).sort_values("score")
print("Top 10 parameter sets (lower score is better):")
display(sweep.head(10))

Top 10 parameter sets (lower score is better):


,jump_prior,jump_threshold,jump_var_mult,lag1_autocorr,jump_rate,z95_abs,score
19,0.06,0.45,9.0,0.865548,0.009333,1.974007,0.874842
20,0.06,0.45,16.0,0.865797,0.008533,1.972467,0.875146
18,0.06,0.45,4.0,0.866127,0.007633,1.972608,0.875519
9,0.03,0.45,4.0,0.865964,0.003400,1.972686,0.875567
11,0.03,0.45,16.0,0.866185,0.004533,1.973188,0.875727
10,0.03,0.45,9.0,0.866202,0.004767,1.973455,0.875729
0,0.01,0.45,4.0,0.866049,0.001133,1.971369,0.875779
3,0.01,0.55,4.0,0.866049,0.000600,1.971369,0.875806
6,0.01,0.70,4.0,0.866049,0.000300,1.971369,0.875821
13,0.03,0.55,9.0,0.866312,0.003300,1.973439,0.875912
